# Equal-budget comparison: schedule vs drift trigger at the same number of retrains

Runs `equal_budget.py` from the thesis repository on one dataset (label delays 0 and 30 days):
schedules and drift triggers over many settings, then each trigger is paired with a schedule
that has the same training window and **no more** retrains.

**Run it once per dataset** (`DATASET` below), with that dataset added as input:
- `sparkov`: **Credit Card Transactions Fraud Detection Dataset** by `kartik2112`
- `baf`: **Bank Account Fraud Dataset Suite (NeurIPS 2022)** by `sgpjesus`
- `ieee`: competition **IEEE-CIS Fraud Detection** (join the competition first)

Settings: Accelerator **None**, Internet **On**, then **Save Version -> Save & Run All (Commit)**.
Output: `equal_budget_<dataset>.zip`. Expected time on Kaggle: BAF ~1 hour, Sparkov ~4-6 hours.
If a run hits the 12-hour limit, add this notebook's previous output as an input and run again:
finished label delays are restored and skipped.

In [ ]:
# ---- settings ----
DATASET = "sparkov"   # "sparkov", "baf" or "ieee"
REPO_URL = "https://github.com/nishan469/fraud-mlops-thesis.git"

In [ ]:
import glob, os, shutil, subprocess, sys
WORK = "/kaggle/working"
CODE = f"{WORK}/fraud-mlops-thesis"
if not os.path.exists(f"{CODE}/equal_budget.py"):
    if os.path.exists(CODE):
        shutil.rmtree(CODE)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=True)
os.chdir(CODE)
!pip install -q "lightgbm>=4.7" pyarrow
import pandas, lightgbm
print("pandas", pandas.__version__, "| lightgbm", lightgbm.__version__, "| CPUs", os.cpu_count())

In [ ]:
# find the data and prepare it in the common layout
MARKER = {"ieee": "train_transaction.csv", "sparkov": "fraudTrain.csv", "baf": "Base.csv"}[DATASET]
hits = glob.glob(f"/kaggle/input/**/{MARKER}", recursive=True)
assert hits, f"{MARKER} not found: add the {DATASET} data as an input"
IN_DIR = os.path.dirname(hits[0])
if DATASET == "ieee":
    DATA, START = IN_DIR, "0.6"
else:
    DATA, START = f"data/{DATASET}", "0.5"
    if not os.path.exists(f"{DATA}/transactions.parquet"):
        cmd = (["prepare_sparkov.py", "--data_dir", IN_DIR, "--out_dir", DATA] if DATASET == "sparkov"
               else ["prepare_baf.py", "--in_dir", IN_DIR, "--out_dir", DATA])
        subprocess.run([sys.executable, *cmd], check=True)
OUT = f"outputs/{DATASET}/equal_budget"
# restore finished delays from a previous version of this notebook, if attached
prev = glob.glob(f"/kaggle/input/**/outputs/{DATASET}/equal_budget/eb_runs.csv", recursive=True)
if prev:
    shutil.copytree(os.path.dirname(prev[0]), OUT, dirs_exist_ok=True)
    print("Restored", prev[0])
print("Data:", DATA)

In [ ]:
env = {**os.environ, "PYTHONUNBUFFERED": "1"}
p = subprocess.Popen([sys.executable, "-W", "ignore", "equal_budget.py", "--data_dir", DATA,
                      "--stream_start_frac", START, "--out_dir", OUT, "--delays", "0,30", "--resume"],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
for line in p.stdout:
    print(line, end="")
assert p.wait() == 0, "run failed (see above)"

In [ ]:
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv(f"{OUT}/eb_summary.csv"))
display(Image(f"{OUT}/eb_frontier.png"))
import zipfile
dest = f"{WORK}/equal_budget_{DATASET}.zip"
with zipfile.ZipFile(dest, "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(OUT):
        z.write(os.path.join(OUT, f), f"{DATASET}/equal_budget/{f}")
print(dest)